# Phase 6b Stage 2 — train the two ablation finalists

Trains `w125` (P3/P4/P5, width 0.125) and `p34` (P3/P4, width 0.25) from random init and compares accuracy — the piece Phase 6a deliberately left undone (it measured flash/RAM cost only, no training).

**Moved here from a local M4 Pro run** that hung the machine: `--cache ram` decodes this dataset to ~19GB, and on Apple Silicon's unified memory that competes directly with the GPU's own pool. Colab's GPU has dedicated VRAM separate from system RAM, so that failure mode doesn't apply here — but free-tier Colab system RAM (~12GB) is still *smaller* than the Mac's 24GB, so this notebook keeps `--cache` off regardless. Plain disk reads are fast enough on a real GPU.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine). Then run cells top to bottom.

**You'll need:** a Kaggle account + API token (kaggle.com -> Settings -> Create New Token -> downloads `kaggle.json`). Cell 3 will prompt you to upload it.

## 1. Clone the repo and install dependencies

In [1]:
!git clone -b phase/six https://github.com/akash-reddy-k/edgeAi.git
%cd edgeAi
!git log --oneline -3

Cloning into 'edgeAi'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 99 (delta 29), reused 57 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 13.72 MiB | 20.96 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/edgeAi
d9f42c0 (HEAD -> phase/six, origin/phase/six) Created using Colab
bbd4cfd add Colab notebook for Phase 6b Stage 2 training
9398056 train.py — Phase 6b Stage 2 support, device portability


In [2]:
!pip install -q -r requirements.txt kaggle

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime > GPU before continuing")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requ

## 2. Kaggle credentials + dataset download

Regenerating the dataset from source (rather than uploading your local 874MB copy) keeps this reproducible from a clean clone, same as the README's local instructions. `prepare_dataset.py` is deterministic — fixed val-scene split, fixed pseudo-label confidence threshold — so this reproduces the same train/val split and labels your local run used.

In [3]:
!kaggle datasets download -d nikanvasei/shanghaitech-campus-dataset-test -p data/SHANGHAI_Test --unzip

Dataset URL: https://www.kaggle.com/datasets/nikanvasei/shanghaitech-campus-dataset-test
License(s): MIT
100% 8.72G/8.72G [07:22<00:00, 21.2MB/s]



In [4]:
# Kaggle's --unzip sometimes nests one level deep depending on how the
# uploader zipped it. Find frames/ and SHANGHAI_test.txt wherever they
# landed under data/ and normalise to the layout prepare_dataset.py expects.
import shutil
from pathlib import Path

base = Path("data/SHANGHAI_Test")
root = Path("data")

if not (base / "frames").exists():
    candidates = [p for p in root.rglob("frames") if p.is_dir()]
    assert candidates, "No frames/ directory found under data/ — inspect the download manually"
    shutil.move(str(candidates[0]), str(base / "frames"))

if not (base / "SHANGHAI_test.txt").exists():
    candidates = list(root.rglob("SHANGHAI_test.txt"))
    assert candidates, "No SHANGHAI_test.txt found under data/ — inspect the download manually"
    shutil.move(str(candidates[0]), str(base / "SHANGHAI_test.txt"))

n_scenes = len(list((base / "frames").iterdir()))
print(f"OK — {n_scenes} scenes under {base}/frames")

OK — 199 scenes under data/SHANGHAI_Test/frames


## 3. Base weights + pseudo-labeled dataset

`prepare_dataset.py` uses stock `yolov8n.pt` as a pseudo-labeler (COCO -> campus domain adaptation), same as Phase 3 locally.

In [5]:
import os, shutil
from ultralytics import YOLO

os.makedirs("models", exist_ok=True)
YOLO("yolov8n.pt")  # downloads to cwd if not already cached
if os.path.exists("yolov8n.pt") and not os.path.exists("models/yolov8n.pt"):
    shutil.move("yolov8n.pt", "models/yolov8n.pt")
print("models/yolov8n.pt present:", os.path.exists("models/yolov8n.pt"))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
models/yolov8n.pt present: True


In [6]:
!python prepare_dataset.py

Scenes found: 199  |  Val scenes: 8

  [train] 01_001         765 frames  detections: 1249
  [train] 01_0015        433 frames  detections: 1153
  [train] 01_0025        601 frames  detections: 2226
  [train] 01_0027        409 frames  detections: 1944
  [val  ] 01_0028        457 frames  detections: 1248
  [train] 01_004         770 frames  detections: 2708
  [train] 01_0051        337 frames  detections: 841
  [train] 01_0052        337 frames  detections: 3188
  [train] 01_0053        457 frames  detections: 3052
  [val  ] 01_0055        313 frames  detections: 1329
  [train] 01_0056        529 frames  detections: 1957
  [val  ] 01_006         506 frames  detections: 571
  [train] 01_0064        289 frames  detections: 1821
  [train] 01_008        1010 frames  detections: 3025
  [train] 01_012         738 frames  detections: 1380
  [train] 01_0135        409 frames  detections: 550
  [train] 01_0136        529 frames  detections: 2654
  [val  ] 01_0139        217 frames  detections:

## 4. Train the two Stage 2 finalists

Both from random init (`--variant` implies `pretrained=False` in `train.py` — see its docstring for why). `--device 0` targets the Colab GPU. No `--cache`: kept off deliberately, see the caveat at the top.

In [ ]:
!python train.py --variant w125 --epochs 40 --patience 15 --batch 16 --device 0

Phase 6b Stage 2 — training P3/P4/P5  w0.125  (key=w125)  from random init
  epochs=40  patience=15  batch=16  imgsz=640  device=0

Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tr

In [ ]:
!python train.py --variant p34 --epochs 40 --patience 15 --batch 16 --device 0

## 5. Package results for download

Zips both run directories (`results.csv`, `args.yaml`, plots) and the best weights. Unzip locally into the same paths — `training_runs/*` -> `runs/detect/results/training/`, `models/*.pt` -> `models/` — and everything downstream (the Stage 2 comparison, README) reads it exactly like a local run.

In [ ]:
import shutil, os
from google.colab import files

stage_dir = "stage2_package"
os.makedirs(f"{stage_dir}/models", exist_ok=True)
shutil.copytree("runs/detect/results/training", f"{stage_dir}/training_runs", dirs_exist_ok=True)

for f in ["models/yolov8_ablation_w125.pt", "models/yolov8_ablation_p34.pt"]:
    if os.path.exists(f):
        shutil.copy(f, f"{stage_dir}/models/")
    else:
        print(f"WARNING: {f} missing — training may not have completed")

shutil.make_archive("stage2_results", "zip", stage_dir)
files.download("stage2_results.zip")

In [ ]:
# Quick summary printed here too, in case the download step is skipped —
# paste this back into the conversation if you'd rather not transfer files.
import pandas as pd

for variant in ["w125", "p34"]:
    csv_path = f"runs/detect/results/training/yolov8_ablation_{variant}/results.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        last = df.iloc[-1]
        print(f"--- {variant} ---")
        print(f"  epochs run : {len(df)}")
        print(f"  mAP50      : {last['metrics/mAP50(B)']:.4f}")
        print(f"  mAP50-95   : {last['metrics/mAP50-95(B)']:.4f}")
        print(f"  Precision  : {last['metrics/precision(B)']:.4f}")
        print(f"  Recall     : {last['metrics/recall(B)']:.4f}\n")